# Dental charting on panoramic X-rays (DENTEX)

**Goal:** an AI that reads a panoramic X-ray (OPG) the way a dentist charts it, then checks
whether its explanations make anatomical sense and how it compares with a dentist.

| Part | What happens | Time on a free T4 GPU |
|---|---|---|
| 0 | Download the DENTEX data (real OPGs from three institutions) | ~10 min |
| 1 | **Dental chart:** number every tooth (FDI) and list missing teeth | ~35 min |
| 2 | **Findings:** caries, deep caries, periapical lesions, impacted teeth, placed on the chart (e.g. "36 DC") | ~45 min |
| 3 | **Explanation check:** does the heat-map point to the crown for caries and to the apex for periapical lesions? | ~3 min |
| 4 | **Dentist study:** you chart 50 X-rays blinded; dentist vs AI vs ground truth | ~1 h of your charting |
| 5 | Save a results summary (`results.md`, `results.json`) | < 1 min |
| 6 | **Dashboard:** try the AI on your own X-ray in a web page | ~2 min |

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then run the cells one by one
(`Shift + Enter`). Results and trained models are saved to your Google Drive, so if Colab
disconnects you can simply run everything again and training continues where it stopped.

*Data: DENTEX (Hamamci et al., 2023), licence CC BY-NC-SA 4.0 - non-commercial use. Do not
upload the X-rays to GitHub.*

## Setup

In [ ]:
#@title Unpack the project code (run this cell first)
# The dentchart package (data, models, training, evaluation, explanations, dentist study) is
# embedded here so the notebook runs on its own. Readable source: the dentchart/ folder of the project.
import base64, io, os, sys, zipfile
CODE = "..."
ROOT = os.environ.get("DENTCHART_CODE_DIR", "/content/panoramic-dental-charting")
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall(ROOT)
sys.path.insert(0, ROOT)
print("Project code ready in", ROOT)

In [ ]:
import os, sys, subprocess
SMOKE = os.environ.get("DENTCHART_SMOKE") == "1"   # developer test mode on fake data - leave as is
if not SMOKE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools", "huggingface_hub", "streamlit"], check=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none - switch the runtime to a GPU!")

In [ ]:
from pathlib import Path
if SMOKE:
    WORK = Path(os.environ["DENTCHART_WORK"])
else:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive"
    WORK = Path("/content/drive/MyDrive/panoramic-dental-charting")
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "assets").mkdir(exist_ok=True)
DATA = WORK / "dentex" if SMOKE else Path("/content/dentex")
print("Results and models are saved in:", WORK)

### Settings
The defaults are sensible. If you get *CUDA out of memory*, set `batch_size` to 2.

In [ ]:
SEED = 42                       # fixes the train / validation / test split
EPOCHS = 2 if SMOKE else 12
TEETH = dict(max_size=640 if SMOKE else 1333, batch_size=2 if SMOKE else 4, lr=0.005 if SMOKE else 0.01)
FINDINGS = dict(max_size=640 if SMOKE else 1600, batch_size=2, lr=0.005)   # caries need more detail
PRETRAINED = not SMOKE          # start from COCO weights (transfer learning)
NUM_WORKERS = 0 if SMOKE else 2

## Part 0. Download DENTEX

The download is about 11 GB. We keep only the two labelled sets this project uses and the
50 official validation X-rays, then delete the zip:

* **enumeration** - 634 OPGs where *every* tooth is boxed with its FDI number
* **disease** - 705 OPGs where *abnormal* teeth are boxed with FDI number + diagnosis
* **validation** - 50 OPGs with full labels, kept aside for your dentist study

In [ ]:
from dentchart import data
if (DATA / "summary.json").exists():
    print("Already prepared:", (DATA / "summary.json").read_text())
else:
    if SMOKE:
        raw = WORK / "raw"
        subprocess.run([sys.executable, f"{ROOT}/scripts/make_synthetic_dentex.py", "--out", str(raw)], check=True)
        train_zip, val_zip, val_json = raw / "training_data.zip", raw / "validation_data.zip", raw / "validation_triple.json"
    else:
        from huggingface_hub import hf_hub_download
        def get(name):
            return hf_hub_download("ibrahimhamamci/DENTEX", f"DENTEX/{name}", repo_type="dataset",
                                   local_dir="/content/dentex_raw")
        train_zip, val_zip, val_json = get("training_data.zip"), get("validation_data.zip"), get("validation_triple.json")
    summary = data.prepare_dentex(train_zip, DATA, val_zip=val_zip, val_json=val_json)
    print(summary)
    if not SMOKE:
        os.remove(train_zip)    # frees about 11 GB of disk

In [ ]:
import matplotlib.pyplot as plt
from dentchart import fdi, viz

teeth_recs, _ = data.load_records(DATA / "enumeration")
dis_recs, DIAGNOSES = data.load_records(DATA / "disease")
val_recs, _ = data.load_records(DATA / "validation", DIAGNOSES)
print(f"{len(teeth_recs)} X-rays with every tooth numbered | {len(dis_recs)} with findings | "
      f"{len(val_recs)} kept for the dentist study")
print("Diagnosis classes in DENTEX order:", DIAGNOSES)

# one training X-ray of each kind (the 50 study X-rays are never shown before you chart them)
fig, axes = plt.subplots(2, 1, figsize=(14, 12))
ds = data.OPGDataset(teeth_recs[:1], DATA / "enumeration" / "images", "teeth")
boxes, labels, _ = ds.annotations(0)
viz.show_xray(axes[0], ds.image(0).numpy(), boxes.numpy(), [str(fdi.LABEL_TO_TOOTH[int(l)]) for l in labels],
              ["#4e79a7"] * len(labels), "Enumeration set: every tooth with its FDI number")
ds = data.OPGDataset(dis_recs[:1], DATA / "disease" / "images", "findings", DIAGNOSES)
anns = dis_recs[0]["anns"]
viz.show_xray(axes[1], ds.image(0).numpy(), [a["box"] for a in anns],
              [f"{a['fdi']} {fdi.CODES.get(a['diagnosis'], a['diagnosis'])}" for a in anns], [fdi.COLORS.get(a["diagnosis"], "#888888") for a in anns],
              "Disease set: abnormal teeth with FDI number and diagnosis")
fig.tight_layout()
plt.show()

## Part 1. The dental chart: number every tooth

A **Faster R-CNN** detector (ResNet-50 backbone, pre-trained on everyday photos - the same
transfer-learning idea as the MRI project) learns to put a box around each tooth and name it
with its FDI number. 32 classes, one per tooth.

*Dental detail:* during training the X-rays are randomly flipped left-right. A flip moves the
teeth to the other side of the mouth, so tooth numbers are mirrored too (36 becomes 46);
flipping without relabelling would teach the model the wrong side.

The 634 X-rays are split 75 / 10 / 15 % into training, validation (to pick the best epoch)
and test (only used once, for the final scores).

In [ ]:
from dentchart import model as M, train as T, evaluate as E

tr, va, te = data.split_records(teeth_recs, seed=SEED)
img_dir = DATA / "enumeration" / "images"
t_train = data.OPGDataset(tr, img_dir, "teeth", train=True)
t_val, t_test = data.OPGDataset(va, img_dir, "teeth"), data.OPGDataset(te, img_dir, "teeth")
print(f"train {len(t_train)} | validation {len(t_val)} | test {len(t_test)} X-rays")

teeth_model = M.build_model(t_train.num_classes, pretrained=PRETRAINED, max_size=TEETH["max_size"])
teeth_hist = T.train_detector(teeth_model, t_train, t_val, WORK / "runs" / "teeth", epochs=EPOCHS,
                              batch_size=TEETH["batch_size"], lr=TEETH["lr"], num_workers=NUM_WORKERS)

In [ ]:
t_pred = E.predict(teeth_model, t_test, num_workers=NUM_WORKERS)
teeth_coco = E.coco_metrics(t_pred, t_test)
teeth_chart = E.teeth_chart_metrics(t_pred, t_test)
print(f"Tooth detection on the test set:  AP50 {teeth_coco['AP50']:.3f} | AP {teeth_coco['AP']:.3f}")
print(f"Teeth found (IoU >= 0.5):         {teeth_chart['detection_rate']:.1%}")
print(f"... with the correct FDI number:  {teeth_chart['numbering_accuracy']:.1%}")
print(f"Chart correct (present/missing):  {teeth_chart['chart_accuracy']:.1%} of tooth positions")
print(f"Missing teeth recognised:         {teeth_chart['missing_sensitivity']:.1%} "
      f"(of {teeth_chart['truly_missing_teeth']} truly missing teeth)")

k = 0
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})
b, names = viz.teeth_boxes(t_pred[k])
viz.show_xray(axes[0], t_test.image(k).numpy(), b, names, ["#4e79a7"] * len(names), "AI tooth numbering (test X-ray)")
viz.plot_dental_chart(set(), present=E.teeth_present(t_pred[k]), ax=axes[1], title="AI dental chart: grey = missing")
fig.tight_layout()
fig.savefig(WORK / "assets" / "teeth_example.png", dpi=120)
plt.show()

**How to read this:** AP50 is the standard detection score (1.0 = every tooth found and named,
no false boxes). The numbering accuracy shows how often a found tooth gets the right FDI
number - neighbouring premolars and molars are the usual confusions.

## Part 2. Findings: caries, deep caries, periapical lesions, impacted teeth

A second detector learns the four DENTEX diagnoses. Each finding is then placed on the chart
by giving it the number of the tooth box it overlaps (from Part 1), e.g. **36 - deep caries**.

The score threshold for a finding is chosen on the validation X-rays; the test X-rays are
only used for the final numbers. A finding counts as correct only if **both** the tooth number
and the diagnosis are right.

In [ ]:
d_tr, d_va, d_te = data.split_records(dis_recs, seed=SEED)
img_dir = DATA / "disease" / "images"
f_train = data.OPGDataset(d_tr, img_dir, "findings", DIAGNOSES, train=True)
f_val = data.OPGDataset(d_va, img_dir, "findings", DIAGNOSES)
f_test = data.OPGDataset(d_te, img_dir, "findings", DIAGNOSES)
print(f"train {len(f_train)} | validation {len(f_val)} | test {len(f_test)} X-rays")

find_model = M.build_model(f_train.num_classes, pretrained=PRETRAINED, max_size=FINDINGS["max_size"])
find_hist = T.train_detector(find_model, f_train, f_val, WORK / "runs" / "findings", epochs=EPOCHS,
                             batch_size=FINDINGS["batch_size"], lr=FINDINGS["lr"], num_workers=NUM_WORKERS)
fig = viz.plot_history({"teeth": teeth_hist, "findings": find_hist})
fig.savefig(WORK / "assets" / "training_curves.png", dpi=120)
plt.show()

In [ ]:
import pandas as pd

def chart_inputs(records):
    find_ds = data.OPGDataset(records, img_dir, "findings", DIAGNOSES)
    teeth_ds = data.OPGDataset(records, img_dir, "teeth")
    return (find_ds, E.predict(find_model, find_ds, num_workers=NUM_WORKERS),
            E.predict(teeth_model, teeth_ds, num_workers=NUM_WORKERS), [E.gt_findings(r, DIAGNOSES) for r in records])

_, fv, tv, gv = chart_inputs(d_va)
THRESH, val_f1 = E.best_threshold(fv, tv, gv, DIAGNOSES)
print(f"Finding threshold chosen on validation: {THRESH:.2f} (validation F1 {val_f1:.3f})")

f_test, fp_test, tp_test, gt_test = chart_inputs(d_te)
find_coco = E.coco_metrics(fp_test, f_test)
ai_test = [E.chart_findings(f, t, DIAGNOSES, find_thresh=THRESH) for f, t in zip(fp_test, tp_test)]
chart_test = E.chart_scores(ai_test, gt_test, DIAGNOSES)
table = pd.DataFrame(chart_test).T[["TP", "FP", "FN", "precision", "recall", "F1"]]
table["AP50 (box)"] = [find_coco["AP50_per_class"].get(d, float("nan")) for d in DIAGNOSES] + [find_coco["AP50"]]
display(table.round(3))

In [ ]:
k = max(range(len(gt_test)), key=lambda i: len(gt_test[i]))          # a test X-ray with several findings
keep = fp_test[k]["scores"] >= THRESH
teeth_of = E.assign_teeth(fp_test[k]["boxes"][keep], tp_test[k])
dx_of = [DIAGNOSES[int(l) - 1] for l in fp_test[k]["labels"][keep]]
fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={"height_ratios": [3, 1, 1]})
viz.show_xray(axes[0], f_test.image(k).numpy(), fp_test[k]["boxes"][keep],
              [f"{t or '?'} {fdi.CODES.get(d, d)}" for t, d in zip(teeth_of, dx_of)], [fdi.COLORS.get(d, "#888888") for d in dx_of],
              "AI findings on a test X-ray")
viz.plot_dental_chart(ai_test[k], present=E.teeth_present(tp_test[k]), ax=axes[1], title="AI chart")
viz.plot_dental_chart(gt_test[k], ax=axes[2], title="DENTEX ground truth")
fig.tight_layout()
fig.savefig(WORK / "assets" / "findings_example.png", dpi=120)
plt.show()

## Part 3. Explanation check: does the AI look at the right part of the tooth?

For every finding the model detects correctly, a **HiResCAM** heat-map shows which part of the
X-ray drove that detection. HiResCAM is a variant of Grad-CAM that keeps the exact location of
the evidence (it multiplies gradients and features point by point instead of averaging the
gradients first). Then a dental question is asked of each heat-map:

* **caries / deep caries** - the evidence should be in the **crown** half of the tooth
* **periapical lesion** - the evidence should be in the **root** half, towards the apex

Upper teeth point down on an OPG and lower teeth point up, so "crown half" is worked out from
the FDI number. `plausible_rate` is the share of findings where the heat-map is where a
dentist would look (with a 95% confidence interval). Around 50% means no better than chance;
close to 100% means the model consistently looks where a dentist would. Impacted teeth have no
single expected region.

In [ ]:
from dentchart import explain
cam_rows, cam_summary, cam_examples = explain.explanation_check(find_model, f_test, fp_test, score_thresh=THRESH)
if not cam_summary.empty:
    display(cam_summary.round(3))
    fig = viz.plot_cam_examples(cam_examples, f_test)
    fig.savefig(WORK / "assets" / "explanation_check.png", dpi=120)
    plt.show()

**Your expert look:** scroll through the heat-maps above. Do they match what you would look at?
Note examples where the model is right for the wrong reason (for example a heat-map on the
neighbouring restoration) - these observations are valuable for the write-up.

## Part 4. Dentist study: you versus the AI (blinded)

You chart the **50 official DENTEX validation X-rays** yourself. They were never used for
training or shown above. For each X-ray, list every abnormal tooth:

`C` = caries, `DC` = deep caries, `PA` = periapical lesion, `I` = impacted  -  e.g. **`36 DC; 48 I`**
(type **`none`** if the X-ray has no abnormal tooth, then press *Save & next*).

* Chart **before** looking at any AI result for these X-rays.
* Your answers are saved to `my_chart.csv` in your Drive after every X-ray: you can stop and
  continue later (re-run this cell; it opens at the first X-ray not yet charted).
* Prefer a spreadsheet? Fill `my_chart.csv` in Excel instead (`reviewed` = yes for each row).
* Saw AI results for some study X-rays before charting them (for example after a test run)?
  Set `EXCLUDE_SEEN = True` once: those X-rays are left out so the study stays blind.

In [ ]:
from dentchart import agreement as A
STUDY = WORK / "dentist_study"
chart_csv = A.export_study(val_recs, DATA / "validation" / "images", STUDY)
print("Study X-rays and chart:", STUDY)

EXCLUDE_SEEN = False   # True once if you saw AI results for X-rays already marked as charted
marker = STUDY / "excluded_after_unblinding.txt"
if EXCLUDE_SEEN and not marker.exists():
    left_out = A.exclude_reviewed(chart_csv)
    marker.write_text("\n".join(left_out))
    print(f"Left out of the study (AI result already seen): {len(left_out)} X-rays", *left_out)

if SMOKE:   # developer test: a fake dentist who copies part of the truth
    df_chart = pd.read_csv(chart_csv, dtype=str, keep_default_na=False)
    for i, r in enumerate(val_recs):
        truth = sorted(E.gt_findings(r, DIAGNOSES))[: 1 + i % 2]
        df_chart.loc[df_chart.image == r["file_name"], ["findings", "reviewed"]] = [
            "; ".join(f"{t} {fdi.CODES.get(d, d)}" for t, d in truth), "yes"]
    df_chart.to_csv(chart_csv, index=False)
else:
    A.charting_tool(STUDY / "images", chart_csv)

**When all 50 X-rays are charted**, run the next cell. Only now does the AI look at these
X-rays.

In [ ]:
dentist, problems = A.read_chart(chart_csv)
chart_df = pd.read_csv(chart_csv, dtype=str, keep_default_na=False)
todo = int((~chart_df["reviewed"].str.lower().isin(["yes", "excluded"])).sum())
excluded = int((chart_df["reviewed"].str.lower() == "excluded").sum())
print(f"{len(dentist)} X-rays charted, {todo} still to chart, {excluded} left out", *problems, sep="\n")
STUDY_DONE = todo == 0 and len(dentist) > 0

if STUDY_DONE:
    v_dir = DATA / "validation" / "images"
    v_find = data.OPGDataset(val_recs, v_dir, "findings", DIAGNOSES)
    v_teeth = data.OPGDataset(val_recs, v_dir, "teeth")
    vf = E.predict(find_model, v_find, num_workers=NUM_WORKERS)
    vt = E.predict(teeth_model, v_teeth, num_workers=NUM_WORKERS)
    ai = {r["file_name"]: E.chart_findings(f, t, DIAGNOSES, find_thresh=THRESH) for r, f, t in zip(val_recs, vf, vt)}
    gt = {r["file_name"]: E.gt_findings(r, DIAGNOSES) for r in val_recs}
    study_scores, study_pairs = A.compare(gt, dentist, ai, DIAGNOSES)
    display(study_scores.round(3))
    display(study_pairs.round(3))
    fig = viz.plot_agreement(study_scores)
    fig.savefig(WORK / "assets" / "dentist_vs_ai.png", dpi=120)
    plt.show()
else:
    print("Results stay hidden until every study X-ray is charted, so the study stays blind.")
    study_scores, study_pairs = A.compare({}, {}, {}, DIAGNOSES)

In [ ]:
# Second-reader view: true findings the AI caught that you missed, and the reverse
if not STUDY_DONE:
    print("Shown once every study X-ray is charted.")
else:
    for img in sorted(dentist):
        ai_only = (ai[img] & gt[img]) - dentist[img]
        you_only = (dentist[img] & gt[img]) - ai[img]
        if ai_only or you_only:
            fmt = lambda s: ", ".join(f"{t} {fdi.CODES.get(d, d)}" for t, d in sorted(s)) or "-"
            print(f"{img}:  AI caught {fmt(ai_only)}  |  you caught {fmt(you_only)}")

**How to read this:** *sensitivity* = share of true findings found, *PPV* = share of charted
findings that are true, *kappa* = agreement beyond chance (0.6-0.8 substantial, > 0.8 almost
perfect). *Dentist + AI* shows what happens when the AI is used as a second reader. Remember
that DENTEX labels were also made by dentists: where you disagree with the "truth", review those
X-rays for your write-up (without changing your chart) - some disagreements may be label errors,
which is a finding in itself.

## Part 5. Save a results summary

In [ ]:
import json
results = {
    "teeth": {**{k: teeth_coco[k] for k in ("AP", "AP50", "AP75")}, **teeth_chart},
    "findings": {"threshold": THRESH, "AP50": find_coco["AP50"], "AP50_per_class": find_coco["AP50_per_class"],
                 "chart": chart_test},
    "explanation_check": cam_summary.to_dict(orient="records"),
    "dentist_study": study_scores.to_dict(orient="records") if STUDY_DONE else "not completed",
    "dentist_vs_ai": study_pairs.to_dict(orient="records") if STUDY_DONE else "not completed",
    "split_seed": SEED, "epochs": EPOCHS,
}
(WORK / "results.json").write_text(json.dumps(results, indent=2, default=float))

pct = lambda v: f"{100 * v:.1f}%"
lines = ["| Part | Result |", "|---|---|",
         f"| Tooth detection (test) | AP50 {teeth_coco['AP50']:.3f} |",
         f"| Correct FDI number | {pct(teeth_chart['numbering_accuracy'])} of found teeth |",
         f"| Missing teeth recognised | {pct(teeth_chart['missing_sensitivity'])} |",
         f"| Findings on the chart (test) | F1 {chart_test['All']['F1']:.3f} (precision {chart_test['All']['precision']:.3f}, recall {chart_test['All']['recall']:.3f}) |"]
for row in cam_summary.to_dict(orient="records"):
    if row["expected_region"] != "-":
        lines.append(f"| Heat-map in the {row['expected_region']} - {row['diagnosis']} | {pct(row['plausible_rate'])} of {row['findings']} findings (95% CI {row['plausible_95CI']}) |")
if cam_summary.empty:
    lines.append("| Explanation check | no heat-maps yet - re-run Part 3 |")
if STUDY_DONE:
    for rater in ("Dentist", "AI", "Dentist + AI"):
        r = study_scores[(study_scores.diagnosis == "All") & (study_scores.rater == rater)].iloc[0]
        lines.append(f"| {rater} on {len(dentist)} study X-rays | sensitivity {pct(r['sensitivity'])} (95% CI {r['sens_95CI']}), PPV {pct(r['PPV'])} |")
else:
    lines.append("| Dentist study | not completed yet |")
summary_md = "\n".join(lines)
(WORK / "results.md").write_text(summary_md)
print(summary_md)
print("\nSaved:", WORK / "results.json", "and the figures in", WORK / "assets")

**Next step:** send `results.md` and the figures in `assets/` (both in your Drive folder), and
the README on GitHub can be updated with your real results.

## Part 6. Try it on your own X-ray: the dashboard

A small web page where you upload a panoramic X-ray and get the tooth numbers, the findings and
the dental chart (`app/dashboard.py`). It is made for X-rays the models have not seen
(`dentchart/infer.py`):

* the trained models are loaded strictly: a wrong path or file is an error, never a silently
  untrained model (which produces convincing-looking nonsense);
* dark or flat strips at the edges (a scanner's label band, white frames) are cropped;
* if the X-ray is cropped, squeezed or very different in size, its scale is fitted to the
  training X-rays (the setting where the tooth model finds the most teeth is used);
* a finding is charted only if it lies on a detected tooth.

First a check: a DENTEX test X-ray, and a copy made to look like a screenshot from another system
(black label band added, squeezed sideways), both compared with the DENTEX labels.

In [ ]:
from dentchart import infer
_ = infer.check_on_dentex(DATA, [WORK])

Start the dashboard. Colab's own port links do not carry Streamlit's live connection, so a free
Cloudflare tunnel gives it a temporary web address. **Anyone with the link can open the page
while it runs: upload only anonymised X-rays, do not share the link, and stop the dashboard
(next cell) when you are done.**

In [ ]:
import re, subprocess, time, urllib.request
LOGS = WORK if SMOKE else Path("/content")

def stop_dashboard():
    subprocess.run("pkill -f '[s]treamlit run'; pkill -f '[c]loudflared tunnel'", shell=True)

def answers(url, seconds=60):
    for _ in range(seconds):
        try:
            if urllib.request.urlopen(url, timeout=2).read() == b"ok":
                return True
        except OSError:
            pass
        time.sleep(1)
    return False

stop_dashboard()
subprocess.Popen(["streamlit", "run", f"{ROOT}/app/dashboard.py", "--server.port", "8501",
                  "--server.headless", "true", "--browser.gatherUsageStats", "false"],
                 env=dict(os.environ, DENTCHART_MODEL_DIRS=str(WORK)),
                 stdout=open(LOGS / "streamlit.log", "w"), stderr=subprocess.STDOUT)
if not answers("http://localhost:8501/_stcore/health"):
    raise RuntimeError(f"The dashboard did not start: see {LOGS / 'streamlit.log'}")

if SMOKE:
    print("Dashboard running (test mode, no public link)")
else:
    if not Path("/content/cloudflared").exists():
        subprocess.run(["wget", "-q", "-O", "/content/cloudflared", "https://github.com/cloudflare/cloudflared/"
                        "releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod("/content/cloudflared", 0o755)
    subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                     stdout=open(LOGS / "cloudflared.log", "w"), stderr=subprocess.STDOUT)
    link = None
    for _ in range(60):
        time.sleep(1)
        found = re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", (LOGS / "cloudflared.log").read_text())
        if found:
            link = found[0]
            break
    print("Open the dashboard:", link or f"no link yet - see {LOGS / 'cloudflared.log'}")

In [ ]:
stop_dashboard()   # when you are done: stops the dashboard and closes the link